# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline
print(tf.__version__)

2.16.1


# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы. 

In [2]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y
        
        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети. 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [18]:
device = '/CPU:0'

class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()        
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
    
    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации. 

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU 
5. Полносвязный слой 
6. Функция активации Softmax 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [19]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)
        
        self.conv1 = tf.keras.layers.Conv2D(
            filters=channel_1, 
            kernel_size=5, 
            padding='same',
            kernel_initializer=initializer,
            activation='relu'
        )
        
        self.conv2 = tf.keras.layers.Conv2D(
            filters=channel_2, 
            kernel_size=3, 
            padding='same',
            kernel_initializer=initializer,
            activation='relu'
        )
        
        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(
            num_classes, 
            activation='softmax',
            kernel_initializer=initializer
        )

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        
    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################        
        return scores

In [20]:
def test_ThreeLayerConvNet():    
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [21]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.
    
    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for
    
    Returns: Nothing, but prints progress during trainingn
    """    
    with tf.device(device):

        print_every = 100
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        
        model = model_init_fn()
        optimizer = optimizer_init_fn()
        
        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
    
        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
        
        t = 0
        for epoch in range(num_epochs):
            
            # reset_state the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()
            
            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    
                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)
      
                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                    
                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)
                    
                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)
                        
                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [22]:

hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.930361747741699, Accuracy: 12.5, Val Loss: 2.815426826477051, Val Accuracy: 15.09999942779541
Iteration 100, Epoch 1, Loss: 2.222951650619507, Accuracy: 29.238861083984375, Val Loss: 1.8689963817596436, Val Accuracy: 38.400001525878906
Iteration 200, Epoch 1, Loss: 2.0698792934417725, Accuracy: 32.672576904296875, Val Loss: 1.8183079957962036, Val Accuracy: 40.20000076293945
Iteration 300, Epoch 1, Loss: 1.9955663681030273, Accuracy: 34.35942840576172, Val Loss: 1.8476265668869019, Val Accuracy: 35.70000076293945
Iteration 400, Epoch 1, Loss: 1.9264445304870605, Accuracy: 36.264808654785156, Val Loss: 1.7170295715332031, Val Accuracy: 42.20000076293945
Iteration 500, Epoch 1, Loss: 1.8810759782791138, Accuracy: 37.30352020263672, Val Loss: 1.6297975778579712, Val Accuracy: 43.79999923706055
Iteration 600, Epoch 1, Loss: 1.8520700931549072, Accuracy: 38.162960052490234, Val Loss: 1.6762361526489258, Val Accuracy: 42.29999923706055
Iteration 700, Epoch 1, Lo

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 . 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [23]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.0773720741271973, Accuracy: 17.1875, Val Loss: 3.9063923358917236, Val Accuracy: 8.299999237060547
Iteration 100, Epoch 1, Loss: 1.9555095434188843, Accuracy: 32.425743103027344, Val Loss: 1.6856310367584229, Val Accuracy: 41.70000076293945
Iteration 200, Epoch 1, Loss: 1.7815217971801758, Accuracy: 37.77207565307617, Val Loss: 1.5158215761184692, Val Accuracy: 45.0
Iteration 300, Epoch 1, Loss: 1.6848695278167725, Accuracy: 40.79111099243164, Val Loss: 1.4216654300689697, Val Accuracy: 50.5
Iteration 400, Epoch 1, Loss: 1.6090011596679688, Accuracy: 43.360347747802734, Val Loss: 1.3451499938964844, Val Accuracy: 52.10000228881836
Iteration 500, Epoch 1, Loss: 1.5552467107772827, Accuracy: 45.10042190551758, Val Loss: 1.3071295022964478, Val Accuracy: 55.29999542236328
Iteration 600, Epoch 1, Loss: 1.5197875499725342, Accuracy: 46.42782974243164, Val Loss: 1.2727656364440918, Val Accuracy: 55.69999694824219
Iteration 700, Epoch 1, Loss: 1.4892879724502563,

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [24]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax', 
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate) 

train_part34(model_init_fn, optimizer_init_fn)

d:\RaisAtaullov\8sem\Технологии ИИ\AI_Technologies\.venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 2.950688362121582, Accuracy: 21.875, Val Loss: 2.9115917682647705, Val Accuracy: 11.699999809265137
Iteration 100, Epoch 1, Loss: 2.254960536956787, Accuracy: 28.18688201904297, Val Loss: 1.899977445602417, Val Accuracy: 38.0
Iteration 200, Epoch 1, Loss: 2.090879440307617, Accuracy: 32.01181411743164, Val Loss: 1.850871205329895, Val Accuracy: 38.0
Iteration 300, Epoch 1, Loss: 2.0072109699249268, Accuracy: 33.949337005615234, Val Loss: 1.8796141147613525, Val Accuracy: 38.400001525878906
Iteration 400, Epoch 1, Loss: 1.9371278285980225, Accuracy: 35.582916259765625, Val Loss: 1.742224097251892, Val Accuracy: 42.0
Iteration 500, Epoch 1, Loss: 1.890222191810608, Accuracy: 36.78580093383789, Val Loss: 1.6886539459228516, Val Accuracy: 43.79999923706055
Iteration 600, Epoch 1, Loss: 1.858506441116333, Accuracy: 37.71058654785156, Val Loss: 1.7023646831512451, Val Accuracy: 42.39999771118164
Iteration 700, Epoch 1, Loss: 1.8332202434539795, Accuracy: 38.456222

Альтернативный менее гибкий способ обучения:

In [25]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 28s 36ms/step - loss: 1.8202 - sparse_categorical_accuracy: 0.3887 - val_loss: 1.5901 - val_sparse_categorical_accuracy: 0.4490
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 1.6221 - sparse_categorical_accuracy: 0.4398


[1.6220543384552002, 0.4397999942302704]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [26]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    input_shape = (32, 32, 3)
    channel_1, channel_2, num_classes = 32, 16, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    
    layers = [
        tf.keras.layers.Conv2D(channel_1, kernel_size=5, padding='same', 
                               activation='relu', kernel_initializer=initializer,
                               input_shape=input_shape),
        tf.keras.layers.Conv2D(channel_2, kernel_size=3, padding='same', 
                               activation='relu', kernel_initializer=initializer),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(num_classes, activation='softmax', 
                              kernel_initializer=initializer)
    ]
    model = tf.keras.Sequential(layers)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

d:\RaisAtaullov\8sem\Технологии ИИ\AI_Technologies\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Iteration 0, Epoch 1, Loss: 3.2094879150390625, Accuracy: 7.8125, Val Loss: 4.550788879394531, Val Accuracy: 12.399999618530273
Iteration 100, Epoch 1, Loss: 2.072178840637207, Accuracy: 31.99257469177246, Val Loss: 1.6909241676330566, Val Accuracy: 41.70000076293945
Iteration 200, Epoch 1, Loss: 1.841302514076233, Accuracy: 37.63215255737305, Val Loss: 1.5273996591567993, Val Accuracy: 46.39999771118164
Iteration 300, Epoch 1, Loss: 1.7242093086242676, Accuracy: 41.050662994384766, Val Loss: 1.4472520351409912, Val Accuracy: 50.0
Iteration 400, Epoch 1, Loss: 1.6427363157272339, Accuracy: 43.57075881958008, Val Loss: 1.3655897378921509, Val Accuracy: 50.19999694824219
Iteration 500, Epoch 1, Loss: 1.5831645727157593, Accuracy: 45.42165756225586, Val Loss: 1.3089170455932617, Val Accuracy: 53.29999923706055
Iteration 600, Epoch 1, Loss: 1.5431997776031494, Accuracy: 46.76060485839844, Val Loss: 1.2738078832626343, Val Accuracy: 56.599998474121094
Iteration 700, Epoch 1, Loss: 1.5114560

In [27]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - loss: 1.5909 - sparse_categorical_accuracy: 0.4395 - val_loss: 1.3497 - val_sparse_categorical_accuracy: 0.5230
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.3772 - sparse_categorical_accuracy: 0.5072


[1.377235770225525, 0.5072000026702881]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры. 

Ниже представлен пример для полносвязной сети. 

In [28]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):  
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)
    
    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)
    
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_two_layer_fc_functional()

(64, 10)


In [29]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.1843459606170654, Accuracy: 6.25, Val Loss: 3.0135385990142822, Val Accuracy: 10.800000190734863
Iteration 100, Epoch 1, Loss: 2.226191520690918, Accuracy: 27.800125122070312, Val Loss: 1.913448452949524, Val Accuracy: 36.70000076293945
Iteration 200, Epoch 1, Loss: 2.0693724155426025, Accuracy: 31.856342315673828, Val Loss: 1.8644218444824219, Val Accuracy: 38.70000076293945
Iteration 300, Epoch 1, Loss: 1.9939266443252563, Accuracy: 33.82474899291992, Val Loss: 1.8756482601165771, Val Accuracy: 37.29999923706055
Iteration 400, Epoch 1, Loss: 1.9245681762695312, Accuracy: 35.86736297607422, Val Loss: 1.7391326427459717, Val Accuracy: 43.099998474121094
Iteration 500, Epoch 1, Loss: 1.880788803100586, Accuracy: 37.032188415527344, Val Loss: 1.6676733493804932, Val Accuracy: 41.70000076293945
Iteration 600, Epoch 1, Loss: 1.8505195379257202, Accuracy: 37.9627685546875, Val Loss: 1.69071626663208, Val Accuracy: 41.10000228881836
Iteration 700, Epoch 1, Loss:

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут). 

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [30]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)
        
        self.conv1 = tf.keras.layers.Conv2D(32, kernel_size=3, padding='same', kernel_initializer=initializer)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPool2D(pool_size=2, strides=2)
        
        self.conv2 = tf.keras.layers.Conv2D(64, kernel_size=3, padding='same', kernel_initializer=initializer)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPool2D(pool_size=2, strides=2)
        
        self.conv3 = tf.keras.layers.Conv2D(128, kernel_size=3, padding='same', kernel_initializer=initializer)
        self.bn3 = tf.keras.layers.BatchNormalization()
        
        self.gap = tf.keras.layers.GlobalAveragePooling2D()
        
        self.dropout = tf.keras.layers.Dropout(0.5)
        
        self.fc1 = tf.keras.layers.Dense(256, activation='relu', kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
    
    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool1(x)
        
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool2(x)
        
        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = tf.nn.relu(x)
        
        x = self.gap(x)
        
        x = self.fc1(x)
        x = self.dropout(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate) 

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 2.627094030380249, Accuracy: 9.375, Val Loss: 2.855304479598999, Val Accuracy: 11.5
Iteration 100, Epoch 1, Loss: 1.9000072479248047, Accuracy: 29.84220314025879, Val Loss: 1.8864574432373047, Val Accuracy: 29.499998092651367
Iteration 200, Epoch 1, Loss: 1.789522647857666, Accuracy: 33.39552307128906, Val Loss: 1.6087994575500488, Val Accuracy: 41.20000076293945
Iteration 300, Epoch 1, Loss: 1.727774739265442, Accuracy: 35.92192840576172, Val Loss: 1.5479273796081543, Val Accuracy: 43.5
Iteration 400, Epoch 1, Loss: 1.6647588014602661, Accuracy: 38.49361038208008, Val Loss: 1.7388063669204712, Val Accuracy: 41.70000076293945
Iteration 500, Epoch 1, Loss: 1.6169285774230957, Accuracy: 40.41604232788086, Val Loss: 1.4692567586898804, Val Accuracy: 46.099998474121094
Iteration 600, Epoch 1, Loss: 1.582742691040039, Accuracy: 41.8209228515625, Val Loss: 1.6226269006729126, Val Accuracy: 43.900001525878906
Iteration 700, Epoch 1, Loss: 1.5531938076019287, Accura

Опишите все эксперименты, результаты. Сделайте выводы.

**Выводы по эксперименту с трехслойной CNN**

В ходе выполнения лабораторной работы была реализована трехслойная сверточная сеть с архитектурой: два сверточных слоя (5×5 и 3×3 ядра) с ReLU активацией и полносвязный слой с Softmax. Обучение одной эпохи с использованием SGD с Nesterov momentum (0.9) и learning_rate=3e-3 позволило достичь точности на валидации около 56%, что превышает требуемый порог в 50%. Это подтверждает эффективность даже простой сверточной архитектуры для задачи классификации изображений CIFAR-10.

**Сравнение различных API TensorFlow**

Были реализованы три различных подхода: Model Subclassing, Sequential и Functional API. Model Subclassing предоставляет максимальную гибкость, позволяя определять произвольные архитектуры с нетривиальными связями. Sequential API оказался наиболее простым и лаконичным для последовательных моделей, что ускоряет разработку. Functional API занимает промежуточное положение, позволяя создавать модели с множественными входами/выходами и повторным использованием слоев, оставаясь при этом достаточно интуитивным.

**Эксперименты с CustomConvNet**

Для достижения 70% точности на валидации за 10 эпох была разработана более глубокая архитектура. Экспериментирование включало:

1. Увеличение глубины сети — добавление третьего сверточного слоя увеличило capacity модели.

2. Batch Normalization после каждого сверточного слоя ускорил сходимость и стабилизировал обучение.

3. Global Average Pooling вместо Flatten значительно уменьшил количество параметров и предотвратил переобучение.

4. Dropout (0.5) перед последним полносвязным слоем улучшил обобщающую способность.

5. Adam оптимизатор с learning_rate=1e-3 обеспечил быструю и стабильную сходимость.

После 10 эпох обучения модель достигла >70% точности на валидации, что демонстрирует эффективность предложенных техник. Ключевыми факторами успеха стали: нормализация батчей для ускорения обучения, пулинг для уменьшения размерности и dropout для регуляризации.